# 🚀 GPU-Accelerated Image Processing
## GPU Specialization Capstone Project
### Author: **PRASIDHA A**

---

This notebook demonstrates GPU-accelerated image processing using:
- **PyTorch CUDA** for tensor-level parallelism
- **CuPy** for raw CUDA kernel execution (when available)
- **4 GPU Kernels**: Gaussian Blur, Sobel Edge Detection, Histogram Equalization, Bilateral Filter
- **CPU vs GPU Benchmark** comparison

> ⚠️ **IMPORTANT**: Make sure you are using a GPU runtime!
> Go to **Runtime → Change runtime type → T4 GPU**

## Step 0: Verify GPU Runtime

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')
    print(f'Streaming Multiprocessors: {props.multi_processor_count}')
    print(f'CUDA Compute Capability: {props.major}.{props.minor}')
else:
    print('⚠️  No GPU found — go to Runtime → Change runtime type → GPU')

## Step 1: Clone Repository & Install Dependencies

In [ ]:
# Clone the project repository
!git clone https://github.com/YOUR_USERNAME/gpu-image-processing-capstone.git
%cd gpu-image-processing-capstone

# Install dependencies
!pip install -q Pillow numpy matplotlib torch torchvision

# Install CuPy for raw CUDA kernels (match your CUDA version)
!pip install -q cupy-cuda12x  # For CUDA 12.x (Colab default as of 2024)
print('Installation complete!')

## Step 2: Import Libraries & Setup

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import time
import json
import os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

try:
    import cupy as cp
    CUPY_AVAILABLE = True
    print('✅ CuPy available — raw CUDA kernels enabled')
except ImportError:
    CUPY_AVAILABLE = False
    print('⚠️  CuPy not available — using PyTorch CUDA backend')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\n🖥️  Active device: {device}')

## Step 3: Load the Main Processing Module

In [ ]:
import sys
sys.path.insert(0, 'src')
from gpu_image_processor import (
    gpu_gaussian_blur,
    gpu_sobel_edges,
    gpu_histogram_equalization,
    gpu_bilateral_filter,
    make_synthetic_image,
    benchmark,
    get_device
)
print('✅ Module loaded successfully')

## Step 4: Generate or Load Test Image

In [ ]:
# Option A: Generate synthetic test image
image = make_synthetic_image(size=512)
print(f'Synthetic image created: {image.shape}')

# Option B: Upload your own image
# from google.colab import files
# uploaded = files.upload()
# fname = list(uploaded.keys())[0]
# image = np.array(Image.open(fname).convert('RGB').resize((512, 512)))

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title('Input Image', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'Shape: {image.shape}, dtype: {image.dtype}')

## Step 5: Run All GPU Kernels

In [ ]:
os.makedirs('outputs', exist_ok=True)
timings = {}

# --- Kernel 1: Gaussian Blur ---
print('Running Kernel 1: Gaussian Blur...')
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0 = time.perf_counter()
blurred = gpu_gaussian_blur(image, kernel_size=15, sigma=3.0, device=device)
torch.cuda.synchronize() if torch.cuda.is_available() else None
timings['Gaussian Blur'] = (time.perf_counter() - t0) * 1000
Image.fromarray(blurred).save('outputs/out_gaussian_blur.png')
print(f'  ✅ Done in {timings["Gaussian Blur"]:.2f} ms')

# --- Kernel 2: Sobel Edge Detection ---
print('Running Kernel 2: Sobel Edge Detection...')
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0 = time.perf_counter()
edges = gpu_sobel_edges(image, device=device)
torch.cuda.synchronize() if torch.cuda.is_available() else None
timings['Sobel Edges'] = (time.perf_counter() - t0) * 1000
Image.fromarray(edges).save('outputs/out_sobel_edges.png')
print(f'  ✅ Done in {timings["Sobel Edges"]:.2f} ms')

# --- Kernel 3: Histogram Equalization ---
print('Running Kernel 3: Histogram Equalization...')
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0 = time.perf_counter()
heq = gpu_histogram_equalization(image, device=device)
torch.cuda.synchronize() if torch.cuda.is_available() else None
timings['Hist. Equalization'] = (time.perf_counter() - t0) * 1000
Image.fromarray(heq).save('outputs/out_hist_eq.png')
print(f'  ✅ Done in {timings["Hist. Equalization"]:.2f} ms')

# --- Kernel 4: Bilateral Filter ---
print('Running Kernel 4: Bilateral Filter...')
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0 = time.perf_counter()
bilat = gpu_bilateral_filter(image, d=7, device=device)
torch.cuda.synchronize() if torch.cuda.is_available() else None
timings['Bilateral Filter'] = (time.perf_counter() - t0) * 1000
Image.fromarray(bilat).save('outputs/out_bilateral.png')
print(f'  ✅ Done in {timings["Bilateral Filter"]:.2f} ms')

print('\n📊 Timing Summary:')
for k, v in timings.items():
    print(f'  {k}: {v:.2f} ms')

## Step 6: Visualize All Results

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('GPU-Accelerated Image Processing — LAKSHMIDHAR N', fontsize=18, fontweight='bold')

images_to_show = [
    (image, 'Original Input', 'viridis'),
    (blurred, 'Gaussian Blur (GPU)', 'viridis'),
    (edges, 'Sobel Edge Detection (GPU)', 'gray'),
    (heq, 'Histogram Equalization (GPU)', 'gray'),
    (bilat, 'Bilateral Filter (GPU)', 'viridis'),
]

for idx, (img_arr, title, cmap) in enumerate(images_to_show):
    r, c = divmod(idx, 3)
    ax = axes[r][c]
    ax.imshow(img_arr, cmap=cmap if img_arr.ndim == 2 else None)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

# Timing bar chart in last panel
ax = axes[1][2]
kernels = list(timings.keys())
vals = list(timings.values())
bars = ax.barh(kernels, vals, color=['#0D9488', '#14B8A6', '#5EEAD4', '#0891B2'])
ax.set_xlabel('Time (ms)', fontsize=11)
ax.set_title('GPU Kernel Execution Times', fontsize=13, fontweight='bold')
for bar, val in zip(bars, vals):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}ms', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('outputs/results_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Results saved to outputs/results_grid.png')

## Step 7: CPU vs GPU Benchmark

In [ ]:
if torch.cuda.is_available():
    print('Running CPU vs GPU benchmark (5 runs each)...\n')
    bench = benchmark(image, n_runs=5)

    # Parse results
    kernel_names = ['gaussian_blur', 'sobel_edges', 'hist_eq']
    cpu_times = [bench.get(f'{k}_cpu', 0) for k in kernel_names]
    gpu_times = [bench.get(f'{k}_gpu', 0) for k in kernel_names]
    speedups = [c/g if g > 0 else 1 for c, g in zip(cpu_times, gpu_times)]

    print('\n📊 Speedup Summary:')
    for name, cpu, gpu, sp in zip(kernel_names, cpu_times, gpu_times, speedups):
        print(f'  {name}: CPU={cpu:.2f}ms | GPU={gpu:.2f}ms | Speedup={sp:.1f}x')

    # Plot
    x = np.arange(len(kernel_names))
    width = 0.35
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('CPU vs GPU Performance Comparison', fontsize=14, fontweight='bold')

    ax1.bar(x - width/2, cpu_times, width, label='CPU', color='#94A3B8')
    ax1.bar(x + width/2, gpu_times, width, label='GPU', color='#0D9488')
    ax1.set_xticks(x)
    ax1.set_xticklabels(['Gaussian\nBlur', 'Sobel\nEdges', 'Hist\nEq.'])
    ax1.set_ylabel('Time (ms)')
    ax1.set_title('Execution Time')
    ax1.legend()

    ax2.bar(kernel_names, speedups, color=['#0D9488', '#14B8A6', '#5EEAD4'])
    ax2.axhline(y=1, color='red', linestyle='--', alpha=0.7, label='1x (no speedup)')
    ax2.set_ylabel('Speedup (x)')
    ax2.set_title('GPU Speedup Factor')
    ax2.set_xticklabels(['Gaussian\nBlur', 'Sobel\nEdges', 'Hist\nEq.'])
    for i, sp in enumerate(speedups):
        ax2.text(i, sp + 0.1, f'{sp:.1f}x', ha='center', fontweight='bold')

    plt.tight_layout()
    plt.savefig('outputs/benchmark_results.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Save log
    with open('outputs/benchmark_log.json', 'w') as f:
        json.dump(bench, f, indent=2)
    print('\n✅ Benchmark saved to outputs/benchmark_results.png')
else:
    print('⚠️  GPU not available. Enable T4 GPU in runtime settings.')

## Step 8: Run via CLI (Optional)

In [ ]:
# Run all kernels via CLI
!python src/gpu_image_processor.py --kernel all --benchmark --output outputs/cli_run
print('\n✅ CLI execution complete. Check outputs/cli_run/')

## Step 9: List All Output Files

In [ ]:
print('📁 Output files:')
for f in sorted(os.listdir('outputs')):
    fpath = os.path.join('outputs', f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {f} ({size_kb:.1f} KB)')

print('\n✅ Capstone project complete!')
print('Author: LAKSHMIDHAR N')
print('GPU Specialization — Final Capstone Project')